# Demo of the Library Sandbox

In [ ]:
from src.sum_impact_assessment.utils.tools import load_living_labs_from_file, load_measures_from_file, load_kpis_from_file, load_kpi_groups_from_file
from src.sum_impact_assessment.models import KPIImpactAnalyzer, PrometheeGaiaAnalyzer
from src.sum_impact_assessment.schemas.mcda import Goal, Alternative
from src.sum_impact_assessment.schemas.impact_analysis import KPIGroupImpactOutput
from src.sum_impact_assessment.schemas.mcda import MCDAAnalysisOutput


from pandas import DataFrame, concat

measures = load_measures_from_file('data/measures.json')
kpis = load_kpis_from_file('data/kpis.json')
kpi_groups = load_kpi_groups_from_file(
    'data/kpi_groups_mcda_goals.json', kpi_definitions=kpis)
living_labs = load_living_labs_from_file(
    'data/living_labs_data.json', kpi_definitions=kpis)


print("KPIs", "-" * 40)
print("Number of kpis : ", len(kpis))
print("Number of kpis groups: ", len(kpi_groups))

print("Number of living labs : ", len(living_labs))
for lab in living_labs:
    print(f"ID: {lab.id}")
    print(f"Name: {lab.name}")
    print(f"Number of KPIs: {len(lab.kpis)}")
    print(f"Number of Measures: {len(lab.measures)}")
    print("-" * 40)

analyzer = KPIImpactAnalyzer(living_labs, measures, kpis, kpi_groups)

final_df = DataFrame()
rreg_results: list[KPIGroupImpactOutput] = []
for group in kpi_groups:
    try:
        print(f"RUNNING ANALYSIS FOR KPI GROUP: {group.name} (ID: {group.id})")
        group_results = analyzer.run_analysis_group(group)
        rreg_results.append(group_results)
        df = DataFrame([coef.model_dump()
                       for coef in group_results.measure_coefficients])

        # add df to final_df
        final_df = concat([final_df, df], ignore_index=True)
    except Exception as e:
        print(
            f"ANALYSIS FAILED FOR KPI GROUP: {group.name} (ID: {group.id}) : {e}")
        continue

print("SAVING RESULTS TO FILES")
file_dir = 'data/output'
print(final_df)
final_df.to_csv(f"{file_dir}/final_measure_coefficients.csv", index=False)

# PROMETHEE GAIA VISUALIZATION
goals = [
    Goal(name=rreg_results[0].name, weight=0.33,
         direction="max", Q=0.0005, S=0.003, P=0.01, F='t5'),
    Goal(name=rreg_results[1].name, weight=0.33,
         direction="max", Q=0.0005, S=0.003, P=0.01, F='t5'),
    Goal(name=rreg_results[2].name, weight=0.34,
         direction="max", Q=0.0005, S=0.003, P=0.01, F='t5')
]
business_alternatives = []

for measure in measures:
    alt = Alternative(
        name=measure.name or f"Measure {measure.id}",
        values={
            rreg_alt.name: next(
                (
                    mcoef.coefficient
                    for mcoef in rreg_alt.measure_coefficients
                    if mcoef.id == measure.id
                ),
                0.0,
            )
            for rreg_alt in rreg_results
        },
    )
    business_alternatives.append(alt)

print("PROMETHEE GAIA INPUT DATA - SAVING TO CSV FILES")
print('goals', goals)
print('alternatives', business_alternatives)

# write goals and alternatives to files CSV for debugging
goals_df = DataFrame([goal.model_dump() for goal in goals])
goals_df.to_csv(f"{file_dir}/promethee_goals.csv", index=False)
# Create a matrix-style dataframe with alternatives as rows and goals as columns
alts_data = []
for alt in business_alternatives:
    row = {'name': alt.name}
    row.update(alt.values)
    alts_data.append(row)

alts_df = DataFrame(alts_data)
alts_df.to_csv(
    f"{file_dir}/promethee_goals_alternatives_matrix.csv", index=False)

analyzer = PrometheeGaiaAnalyzer(
    goals=goals, alternatives=business_alternatives)

# Get complete structured output with standardized keys
print("Running complete PROMETHEE-GAIA analysis with structured output...")
mcda_output: MCDAAnalysisOutput = analyzer.run_analysis(
    run_visualizations=False)

KPIs ----------------------------------------
Number of kpis :  45
Number of kpis groups:  3
Number of living labs :  9
ID: lab_munich
Name: Munich
Number of KPIs: 42
Number of Measures: 14
----------------------------------------
ID: lab_geneva
Name: Geneva
Number of KPIs: 28
Number of Measures: 10
----------------------------------------
ID: lab_jerusalem
Name: Jerusalem
Number of KPIs: 29
Number of Measures: 13
----------------------------------------
ID: lab_athens_penteli
Name: Penteli
Number of KPIs: 34
Number of Measures: 8
----------------------------------------
ID: lab_rotterdam
Name: Rotterdam
Number of KPIs: 30
Number of Measures: 9
----------------------------------------
ID: lab_krakow
Name: Krakow
Number of KPIs: 31
Number of Measures: 9
----------------------------------------
ID: lab_fredrikstad
Name: Fredrikstad
Number of KPIs: 0
Number of Measures: 11
----------------------------------------
ID: lab_larnaca
Name: Larnaca
Number of KPIs: 25
Number of Measures: 4
-----

In [ ]:


print("Running PROMETHEE-GAIA analysis...")
analyzer = PrometheeGaiaAnalyzer(
    goals=goals, alternatives=business_alternatives)
result_pI = analyzer.run_prometheeI(False)
result_pII = analyzer.run_prometheeII(True)
analyzer.run_gaia(25, 25)
result_gaia = analyzer.run_gaia_custom()

print("PROMETHEE I Results:", result_pI)
print("PROMETHEE II Results:", result_pII)
print("GAIA Results:", result_gaia)

print("CUSTOM VISUALIZATIONS, SAMPLES FOR FRONT-END CLIENT")
analyzer.display_gaia()
analyzer.display_prometheeI()
analyzer.display_prometheeII()

# NEW: Unified Structured Output for API/Database

The new `run_analysis()` method returns all MCDA results in a typed, JSON-serializable format perfect for web clients and database storage.

In [2]:
from src.sum_impact_assessment.schemas.mcda import MCDAAnalysisOutput
analyzer = PrometheeGaiaAnalyzer(
    goals=goals, alternatives=business_alternatives)

# Get complete structured output with standardized keys
print("Running complete PROMETHEE-GAIA analysis with structured output...")
mcda_output: MCDAAnalysisOutput = analyzer.run_analysis(run_visualizations=False)

print("\n" + "="*80)
print("STRUCTURED MCDA OUTPUT WITH STANDARDIZED KEYS")
print("="*80)

# Display label mappings
print("\n🏷️  ALTERNATIVE LABELS (Key → Full Name):")
for key, name in list(mcda_output.alternative_labels.items())[:5]:
    print(f"  {key}: {name}")
if len(mcda_output.alternative_labels) > 5:
    print(f"  ... and {len(mcda_output.alternative_labels) - 5} more")

print("\n🏷️  CRITERIA LABELS (Key → Full Name):")
for key, name in mcda_output.criteria_labels.items():
    print(f"  {key}: {name}")

# Display positive flows
print("\n📊 POSITIVE FLOWS (φ+):")
for alt_key, flow in list(mcda_output.positive_flows.items())[:5]:
    alt_name = mcda_output.alternative_labels[alt_key]
    print(f"  {alt_key} ({alt_name}): {flow:.4f}")
if len(mcda_output.positive_flows) > 5:
    print(f"  ... and {len(mcda_output.positive_flows) - 5} more")

# Display negative flows  
print("\n📊 NEGATIVE FLOWS (φ-):")
for alt_key, flow in list(mcda_output.negative_flows.items())[:5]:
    alt_name = mcda_output.alternative_labels[alt_key]
    print(f"  {alt_key} ({alt_name}): {flow:.4f}")
if len(mcda_output.negative_flows) > 5:
    print(f"  ... and {len(mcda_output.negative_flows) - 5} more")

# Display net flows and ranking
print("\n🏆 NET FLOWS & RANKING (φ = φ+ - φ-):")
for rank, alt_key in enumerate(mcda_output.ranking[:10], 1):
    alt_name = mcda_output.alternative_labels[alt_key]
    net_flow = mcda_output.net_flows[alt_key]
    print(f"  #{rank}: {alt_key} ({alt_name}) = {net_flow:.4f}")
if len(mcda_output.ranking) > 10:
    print(f"  ... and {len(mcda_output.ranking) - 10} more")

# Display GAIA info
print(f"\n🎯 GAIA QUALITY: {mcda_output.gaia_quality:.2f}%")
print(f"📍 DECISION STICK: [{mcda_output.gaia_decision_stick[0]:.4f}, {mcda_output.gaia_decision_stick[1]:.4f}]")

print("\n📐 GAIA ALTERNATIVE COORDINATES (using keys):")
for alt in mcda_output.gaia_alternatives[:5]:
    alt_name = mcda_output.alternative_labels[alt.key]
    print(f"  {alt.key} ({alt_name}): x={alt.x:.4f}, y={alt.y:.4f}")
if len(mcda_output.gaia_alternatives) > 5:
    print(f"  ... and {len(mcda_output.gaia_alternatives) - 5} more")

print("\n📐 GAIA CRITERION VECTORS (using keys):")
for crit in mcda_output.gaia_criteria:
    crit_name = mcda_output.criteria_labels[crit.key]
    print(f"  {crit.key} ({crit_name}): x={crit.x:.4f}, y={crit.y:.4f}")

Running complete PROMETHEE-GAIA analysis with structured output...

STRUCTURED MCDA OUTPUT WITH STANDARDIZED KEYS

🏷️  ALTERNATIVE LABELS (Key → Full Name):
  a1: Congestion charges
  a2: Parking charges
  a3: Restricted parking
  a4: Limited traffic zone / Pedestrianisation of streets
  a5: Parking supply management
  ... and 15 more

🏷️  CRITERIA LABELS (Key → Full Name):
  c1: Improve Accessibility
  c2: Improve Safety
  c3: Improve Public Transport

📊 POSITIVE FLOWS (φ+):
  a1 (Congestion charges): 0.1076
  a2 (Parking charges): 0.3980
  a3 (Restricted parking): 0.1216
  a4 (Limited traffic zone / Pedestrianisation of streets): 0.0103
  a5 (Parking supply management): 0.3577
  ... and 15 more

📊 NEGATIVE FLOWS (φ-):
  a1 (Congestion charges): 0.2694
  a2 (Parking charges): 0.0454
  a3 (Restricted parking): 0.2335
  a4 (Limited traffic zone / Pedestrianisation of streets): 0.4658
  a5 (Parking supply management): 0.0618
  ... and 15 more

🏆 NET FLOWS & RANKING (φ = φ+ - φ-):
  #1: a

## JSON Serialization for Database/API

The output can be easily serialized to JSON for storage or API responses:

In [ ]:
import json

# Convert to JSON string
json_output = mcda_output.model_dump_json(indent=2)

print("JSON OUTPUT (first 1000 characters):")
print(json_output[:1000])
print("...")

# Can also save to file
output_file = f"{file_dir}/mcda_complete_output.json"
with open(output_file, 'w') as f:
    f.write(json_output)

print(f"\n✅ Complete MCDA output saved to: {output_file}")

# Show file size
import os
file_size = os.path.getsize(output_file)
print(f"📦 File size: {file_size} bytes ({file_size/1024:.2f} KB)")

# Can be loaded back as Pydantic model
with open(output_file, 'r') as f:
    loaded_data = json.load(f)
    
loaded_output = MCDAAnalysisOutput(**loaded_data)
print(f"\n✅ Successfully loaded and validated from JSON")
print(f"   Ranking: {loaded_output.ranking[:3]}...")

## Data Structure for Front-End Charts

The structured output provides all necessary data for generating charts:

### Chart 1: Net Flow Ranking Bar Chart
- **Data source**: `mcda_output.ranking` (ordered list) and `mcda_output.net_flows` (values)
- **X-axis**: Alternative names from ranking
- **Y-axis**: Net flow values

### Chart 2: Detailed Positive/Negative Flows  
- **Data source**: `mcda_output.positive_flows` and `mcda_output.negative_flows`
- **Display**: Side-by-side or stacked bars showing φ+ and φ- for each alternative

### Chart 3: GAIA Decision Plane (2D Scatter/Vector Plot)
- **Alternatives**: `mcda_output.gaia_alternatives` - plot as scatter points (x, y)
- **Criteria**: `mcda_output.gaia_criteria` - plot as vectors/arrows from origin
- **Decision Stick**: `mcda_output.gaia_decision_stick` - plot as thick arrow from origin
- **Quality indicator**: `mcda_output.gaia_quality` - display as text (e.g., "Quality: 87.5%")

## Example: Prepare Data for Charts

In [ ]:
print("="*80)
print("CHART DATA PREPARATION EXAMPLES")
print("="*80)

# Chart 1: Net Flow Ranking
print("\n📊 Chart 1 Data: Net Flow Ranking")
ranking_chart_data = [
    {
        'key': key,
        'label': mcda_output.alternative_labels[key],
        'value': mcda_output.net_flows[key]
    }
    for key in mcda_output.ranking[:5]  # Top 5
]
print(DataFrame(ranking_chart_data))

# Chart 2: Positive/Negative Flows Comparison
print("\n📊 Chart 2 Data: Positive vs Negative Flows (Top 5)")
flows_chart_data = [
    {
        'key': key,
        'label': mcda_output.alternative_labels[key],
        'positive': mcda_output.positive_flows[key],
        'negative': mcda_output.negative_flows[key],
        'net': mcda_output.net_flows[key]
    }
    for key in mcda_output.ranking[:5]
]
print(DataFrame(flows_chart_data))

# Chart 3: GAIA Plane Data
print("\n📊 Chart 3 Data: GAIA Decision Plane")
print("\nAlternatives (first 5):")
gaia_alts_data = [
    {
        'key': alt.key,
        'label': mcda_output.alternative_labels[alt.key],
        'x': alt.x,
        'y': alt.y
    }
    for alt in mcda_output.gaia_alternatives[:5]
]
print(DataFrame(gaia_alts_data))

print("\nCriteria:")
gaia_criteria_data = [
    {
        'key': crit.key,
        'label': mcda_output.criteria_labels[crit.key],
        'x': crit.x,
        'y': crit.y
    }
    for crit in mcda_output.gaia_criteria
]
print(DataFrame(gaia_criteria_data))

print(f"\nDecision Stick: x={mcda_output.gaia_decision_stick[0]:.4f}, y={mcda_output.gaia_decision_stick[1]:.4f}")
print(f"Quality: {mcda_output.gaia_quality:.2f}%")

print("\n✅ All data is ready for front-end visualization!")